# Revisiting Basics of Spacecraft Kinetics

In [1]:
import numpy as np

import sys
sys.path.insert(0, r"../../")
import AttitudeKinematicsLib as ak

In [2]:
print("Contents of AttitudeKinematicsLib:")
for name in sorted(dir(ak)):
    if not name.startswith("_"):
        print(name)

Contents of AttitudeKinematicsLib:
BInvmat_CRP
BInvmat_EP
BInvmat_Euler
BInvmat_MRP
BInvmat_PRV
Bmat_CRP
Bmat_EP
Bmat_Euler
Bmat_MRP
Bmat_PRV
CRP
CRP_to_DCM
DCM_to_CRP
DCM_to_EP
DCM_to_Euler
DCM_to_MRP
DCM_to_PRV
DCM_utils
EP_to_DCM
EulerAngles
EulerRodriguesParameters
Euler_to_DCM
MRP
MRP_compose
MRP_shadow
MRP_to_DCM
PRV
PRV_to_DCM
integrate_quaternion
normalize_quat
np
quat_derivative
quat_diff
quat_inv
quat_mult
rotation_matrix_x
rotation_matrix_y
rotation_matrix_z
skew_symmetric
solve_ivp
validate_DCM
validate_vec3
validate_vec4


# 1 - General Dynamical System

In [3]:
# Quiz 1 - Question 2
m1 = 1.0    # kg
m2 = 2.0    # kg

R1 = 10.0  # m along n1
R2 = 15.0  # m along n1

F1 = 3.0    # N
F2 = 2.0    # N

R_com = (m1 * R1 + m2 * R2) / (m1 + m2)     # m along n1
a_com = (F1 + F2) / (m1 + m2)               # m/s^2

print("The position of the center of mass is", R_com, "m along n1")
print("The acceleration of the center of mass is", a_com, "m/s^2")

The position of the center of mass is 13.333333333333334 m along n1
The acceleration of the center of mass is 1.6666666666666667 m/s^2


In [4]:
# Quiz 2 - Question 1
m1 = 1.0
m2 = 2.0

v1 = np.array([0.1, 0.2, 0.3])
v2 = np.array([-0.1, 0.2, -0.1])

M = m1 + m2
v_cm = (m1 * v1 + m2 * v2) / M

v1_rel = v1 - v_cm
v2_rel = v2 - v_cm

T_def = 0.5 * m1 * np.dot(v1_rel, v1_rel) + 0.5 * m2 * np.dot(v2_rel, v2_rel)

print("Center of mass velocity:", v_cm)
print("Deformation kinetic energy:", T_def, "J")

Center of mass velocity: [-0.03333333  0.2         0.03333333]
Deformation kinetic energy: 0.06666666666666667 J


In [5]:
# Quiz 3 - Question 1
m1 = 1.0
m2 = 2.0

R1 = np.array([1.0, 2.0, 3.0])
R2 = np.array([-1.0, 2.0, -1.0])
Rp = np.array([1.0, 1.0, 1.0])

v1 = np.array([0.1, 0.2, 0.3])
v2 = np.array([-0.1, 0.2, -0.1])

F1 = np.array([1.0, 1.0, 0.0])
F2 = np.array([0.0, 1.0, 1.0])

r1_rel = R1 - Rp
r2_rel = R2 - Rp

p1 = m1 * v1
p2 = m2 * v2

Hp = np.cross(r1_rel, p1) + np.cross(r2_rel, p2)
Lp = np.cross(r1_rel, F1) + np.cross(r2_rel, F2)

print("Angular momentum about point P:", Hp)
print("Torque about point P:", Lp)



Angular momentum about point P: [ 0.5  0.2 -0.7]
Torque about point P: [ 1.  4. -3.]


# 2 - Single Rigid Body Dynamics

In [6]:
# Quiz 4 - Question 2
m1 = 1.0
m2 = 2.0
M = m1 + m2

R1 = np.array([1.0, 2.0, 3.0])
Rc = np.array([-1.0, 2.0, -1.0])

v1 = np.array([0.1, 0.2, 0.3])
vc = np.array([-0.1, 0.2, 0.0])

R2 = ((M * Rc) - m1 * R1) / m2
Hc = np.cross(Rc, M * vc)

print("R2 =", R2)
print("Hc =", Hc)

R2 = [-2.  2. -3.]
Hc = [0.6 0.3 0. ]


In [7]:
# Quiz 5 - Question 2

I_B = np.diag([100, 150, 200])

sigma_BN = np.array([0.1, 0.2, 0.3])
C_BN = ak.MRP_to_DCM(sigma_BN)
C_NB = C_BN.T  

I_N = C_NB @ I_B @ C_NB.T
print("Inertia matrix in N frame:\n", I_N)

Inertia matrix in N frame:
 [[173.49942464  -5.4244974   28.15725387]
 [ -5.4244974  108.48427828  19.43093791]
 [ 28.15725387  19.43093791 168.01629708]]


In [8]:
# Quiz 5 - Question 4
m = 10

I_B = np.diag([1000, 1500, 2000])

sigma_BN = np.array([0.1, 0.2, 0.3])
C_BN = ak.MRP_to_DCM(sigma_BN)

R_CO_N = np.array([10, 15, 20])

R_CO_B = C_BN @ R_CO_N

R_CO_B_tilde = ak.skew_symmetric(R_CO_B)

I_O_B = I_B + m * (R_CO_B_tilde @ R_CO_B_tilde.T)
print("Inertia matrix about point O in B frame:\n", I_O_B)


Inertia matrix about point O in B frame:
 [[ 7464.79346203 -1040.07819817 -1998.60837452]
 [-1040.07819817  7372.32087227 -2647.34040855]
 [-1998.60837452 -2647.34040855  4162.8856657 ]]


In [9]:
# Quiz 6 - Question 2
I_B = np.diag([100.0, 150.0, 200.0])

sigma_BN = np.array([0.5, 0.5, 0.3])
sigma_GB = np.array([-0.2, 0.3, 0.1])

C_BN = ak.MRP_to_DCM(sigma_BN)
C_GB = ak.MRP_to_DCM(sigma_GB)

omega_BN_G = np.array([0.1, 0.2, 0.3])
L_c_N = np.array([1.0, 1.0, 1.0])

omega_BN_B = C_GB.T @ omega_BN_G
L_c_B = C_BN @ L_c_N

omega_dot_BN_B = np.linalg.inv(I_B) @ (L_c_B - np.cross(omega_BN_B, I_B @ omega_BN_B))

print("omega_BN_B =", omega_BN_B)
print("L_c_B =", L_c_B)
print("B omega_dot_B/N =", omega_dot_BN_B)

omega_BN_B = [ 0.11292705  0.3421976  -0.10073869]
L_c_B = [1.0601242  1.31960761 0.36711364]
B omega_dot_B/N = [ 0.02783751  0.0012133  -0.00782527]


In [10]:
# Quiz 7 - Question 1
omega_B = np.array([0.1, 0.2, 0.3])
L_B = np.array([1.0, -1.0, 0.0])

T_dot = np.dot(omega_B, L_B)

print("T_dot =", T_dot, "W")

T_dot = -0.1 W
